# AeroPulse PM2.5 — Phase 1: Canonical 2–3 Year India Dataset

One-time acquisition, normalization, geography correction, quality control, source adapters, and persistent Parquet datasets.

## Objective

Create the **canonical, reusable AeroPulse PM2.5 base dataset once**.

This notebook deliberately separates:

1. raw acquisition
2. source validation
3. India geography filtering
4. hourly temporal normalization
5. station/sensor quality control
6. PM2.5 target cleaning
7. weather acquisition through a source adapter
8. fire acquisition through a source adapter
9. point-in-time-safe joining
10. dataset quality gates
11. persistence to Parquet

**Important design decision:** OpenAQ remains the PM2.5 target source for this phase. IMD is treated as a meteorological source, not a replacement for the PM2.5 target. The canonical schema is source-neutral so weather can later be swapped from Open-Meteo/reanalysis to IMD without rewriting downstream feature engineering.

## Target dataset contract

Every hourly target record should have:

- `timestamp_utc`
- `timestamp_local`
- `location_id`
- `sensor_id`
- `station_name`
- `country`
- `state`
- `city`
- `lat`
- `lon`
- `pm25`
- `pm25_unit`
- `source`
- `source_quality`
- `coverage_flag`

Weather fields are normalized to:

- temperature
- relative humidity
- surface pressure
- precipitation
- wind speed
- wind direction
- boundary-layer height

Fire fields are normalized to:

- fire count
- FRP
- minimum distance
- multi-radius fire metrics

Downstream notebooks must consume this canonical contract and should never depend on OpenAQ-specific column names.

In [4]:
# ============================================================================
# CONFIGURATION - paths, secrets, acquisition window, tuning knobs
# ============================================================================
# Everything downstream reads CONFIG, so this is the only cell to edit when
# changing scope. Every knob is overridable from .env, so the notebook can be
# re-scoped without touching code.
# ============================================================================
from pathlib import Path
import os, io, json, math, warnings, hashlib, time
import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Load this folder's .env so os.getenv() below can see OPENAQ_API_KEY.
# Without it the OpenAQ adapter sends no auth header and every call 401s.
_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError("No .env found. Create one in this folder with OPENAQ_API_KEY / FIRMS_MAP_KEY.")
load_dotenv(_dotenv_path, override=True)
print("Loaded env from:", _dotenv_path)

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "pm25"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
BASE_DIR = PROCESSED_DIR / "base"
STATION_CACHE = RAW_DIR / "stations"          # per-station parquet, enables resume
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "pm25"
for p in [RAW_DIR, PROCESSED_DIR, BASE_DIR, STATION_CACHE, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # "auto" derives the window from what the sensors ACTUALLY hold (see the
    # manifest cell). OpenAQ's Indian PM2.5 archive does not reach back to
    # 2023 - the median live sensor starts 2025-02-18 - so a hardcoded
    # multi-year window silently yields empty frames.
    "start": os.getenv("PM25_START", "auto"),
    "end": os.getenv("PM25_END", "auto"),
    "country": "IN",

    # Acquisition scope. Cost is roughly MAX_STATIONS x months_in_window API
    # calls, so this is the main runtime dial.
    "max_stations": int(os.getenv("PM25_MAX_STATIONS", "150")),
    # How many locations to resolve sensors for while ranking candidates.
    "max_candidates": int(os.getenv("PM25_MAX_CANDIDATES", "400")),

    # Quality gates.
    "min_station_rows": int(os.getenv("MIN_STATION_ROWS", "1000")),
    "min_coverage": float(os.getenv("MIN_STATION_COVERAGE", "0.20")),

    # Politeness: OpenAQ rate-limits bursts with 429 and emits sporadic 500s.
    "throttle_s": float(os.getenv("PM25_THROTTLE_S", "0.25")),

    # Fire feature geometry.
    "fire_radii_km": [20.0, 50.0, 100.0],
    "fire_window_hours": 24,

    "india_bbox": {"min_lat": 6.0, "max_lat": 37.5, "min_lon": 68.0, "max_lon": 97.5},
    "schema_version": "1.0.0",
}

print("Project  :", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


Loaded env from: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/.env
Project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
Data root: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25
  start: auto
  end: auto
  country: IN
  max_stations: 150
  max_candidates: 400
  min_station_rows: 1000
  min_coverage: 0.2
  throttle_s: 0.25
  fire_radii_km: [20.0, 50.0, 100.0]
  fire_window_hours: 24
  india_bbox: {'min_lat': 6.0, 'max_lat': 37.5, 'min_lon': 68.0, 'max_lon': 97.5}
  schema_version: 1.0.0


In [ ]:
# ============================================================================
# RESUME CHECK - rebuild from cached artifacts instead of refetching
# ============================================================================
# Phase 1's expensive stages are the OpenAQ calls. Once pm25_hourly.parquet and
# station_manifest.parquet exist, the notebook can rebuild everything
# downstream - weather, fire, joins, gates - without touching OpenAQ at all.
#
# This is not just a speed optimisation. It makes the notebook idempotent, and
# it decouples the pipeline from a single API's availability: when the OpenAQ
# key was revoked mid-project, the 1.7M rows already acquired stayed usable
# because Open-Meteo and FIRMS use different credentials.
#
# Set PM25_FORCE_REFETCH=1 in .env to ignore the cache and re-acquire.
# ============================================================================
PM25_FILE = BASE_DIR / "pm25_hourly.parquet"
MANIFEST_FILE = BASE_DIR / "station_manifest.parquet"
FORCE_REFETCH = os.getenv("PM25_FORCE_REFETCH", "0") == "1"
RESUME = PM25_FILE.exists() and MANIFEST_FILE.exists() and not FORCE_REFETCH

if RESUME:
    print("RESUME MODE - reusing cached OpenAQ acquisition:")
    print(f"  {PM25_FILE.name}      {PM25_FILE.stat().st_size/1e6:.1f} MB")
    print(f"  {MANIFEST_FILE.name}  {MANIFEST_FILE.stat().st_size/1e6:.1f} MB")
    print("  Discovery, sensor resolution and hourly acquisition will be SKIPPED.")
    print("  Set PM25_FORCE_REFETCH=1 to re-acquire from OpenAQ.")
else:
    print("FULL ACQUISITION MODE - OpenAQ will be queried "
          f"(force_refetch={FORCE_REFETCH}, cache_present={PM25_FILE.exists()}).")


## 1. Dependencies

Install once in the AeroPulse environment if required:

```bash
python -m pip install pandas numpy pyarrow requests tqdm scikit-learn matplotlib seaborn xarray netcdf4 h5netcdf
```

Optional connectors:

- `openaq` for the official OpenAQ Python client
- `geopandas` / `shapely` for boundary validation
- `xarray` / `netcdf4` for IMD NetCDF products
- satellite/FIRMS client used by the existing AeroPulse code

The notebook intentionally keeps provider-specific logic inside adapter functions.

In [5]:
# ============================================================================
# OPENAQ v3 ADAPTER - authenticated, retrying, correctly paginated
# ============================================================================
# Three defects fixed here, each of which failed silently rather than loudly:
#
#   1. AUTH. The old code did {"X-API-Key": KEY} if KEY else {} - with no key
#      it sent NO auth header, and the API answered 401 Unauthorized, which
#      looks like a bad key rather than a missing one. require_env() now names
#      the real problem.
#   2. RETRIES. OpenAQ returns sporadic 500s and rate-limits with 429. Without
#      retries a multi-thousand-call acquisition loop dies partway through.
#   3. PAGINATION. meta["found"] is an int OR a capped string like ">1000".
#      `len(rows) >= found` raises TypeError against the string form.
# ============================================================================
import requests
from tqdm.auto import tqdm

OPENAQ_API = "https://api.openaq.org/v3"


def require_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(
            f"Missing required environment variable: {name}. "
            "Set it in this folder's .env file."
        )
    return value


def openaq_get(path, params=None, timeout=60, retries=5, backoff=1.5):
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(f"{OPENAQ_API}{path}", params=params or {},
                             headers=headers, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"{r.status_code} from {path}", response=r)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            last = e
            if attempt == retries - 1:
                break
            time.sleep(backoff * (2 ** attempt))
    raise last


def _found_total(meta):
    """meta['found'] is an int, OR a capped string like '>1000'.

    Comparing len(rows) to that string raises TypeError, so a non-int is
    treated as 'total unknown' and we fall back to short-page detection.
    """
    found = meta.get("found")
    return found if isinstance(found, int) else None


def openaq_paginate(path, params=None, max_pages=None):
    params = dict(params or {})
    params.setdefault("limit", 1000)
    limit = params["limit"]
    page, rows = 1, []
    while True:
        params["page"] = page
        payload = openaq_get(path, params)
        batch = payload.get("results", [])
        rows.extend(batch)
        total = _found_total(payload.get("meta", {}))
        # A short page is the reliable end-of-data signal when 'found' is capped.
        if not batch or len(batch) < limit or (total is not None and len(rows) >= total):
            break
        if max_pages and page >= max_pages:
            break
        page += 1
    return rows


def openaq_country_id(iso_code):
    """Resolve an ISO-3166 alpha-2 code to OpenAQ's numeric country id.

    Never hardcode this. The catalog is OpenAQ-internal and unrelated to ISO
    numbering: id 91 is ITALY, id 9 is India. A wrong constant does not error,
    it silently returns a valid dataset for the wrong country.
    """
    for c in openaq_paginate("/countries", {"limit": 200}):
        if str(c.get("code", "")).upper() == iso_code.upper():
            return c["id"]
    raise RuntimeError(f"No OpenAQ country id found for ISO code {iso_code!r}")


def openaq_parameter_id(name):
    for p in openaq_paginate("/parameters", {"limit": 200}):
        if str(p.get("name", "")).lower() == name.lower():
            return p["id"]
    raise RuntimeError(f"No OpenAQ parameter id found for {name!r}")


def location_pm25_sensors(location_id):
    """Every pm25 sensor at a location, with its OWN date range.

    This call is unavoidable. The /locations payload reports datetimeFirst /
    datetimeLast as the max across ALL sensors at the site, so a station whose
    PM2.5 sensor died in 2018 still looks current if its NO2 sensor is live.
    Selecting on location-level dates picks dead sensors and yields zero rows.
    """
    out = []
    for s in openaq_get(f"/locations/{int(location_id)}/sensors").get("results", []):
        if str((s.get("parameter") or {}).get("name", "")).lower() != "pm25":
            continue
        first = pd.to_datetime((s.get("datetimeFirst") or {}).get("utc"), utc=True, errors="coerce")
        last = pd.to_datetime((s.get("datetimeLast") or {}).get("utc"), utc=True, errors="coerce")
        if pd.isna(first) or pd.isna(last):
            continue
        out.append({"sensor_id": s["id"], "sensor_first": first, "sensor_last": last})
    return out


print("OpenAQ adapter ready.")


OpenAQ adapter ready.


## 2. Discover Indian PM2.5 stations

Do not select stations by freshness alone.

A station is eligible for the base dataset only when it passes:

- India geography test
- PM2.5 sensor identification
- sufficient historical coverage
- sufficient number of observations
- stable station coordinates
- plausible PM2.5 values
- no obvious duplicate sensor streams

Persist the station manifest. **Future runs must reuse the manifest**, otherwise station membership changes and model metrics become incomparable.

In [ ]:
# Provider-specific discovery adapter.
#
# FIXED - countries_id was hardcoded to 91, which is ITALY in OpenAQ's catalog;
# India is 9. That constant does not raise an error, it silently returns a valid
# dataset for the wrong country, so both ids are now resolved from the API by
# name and asserted below.

if RESUME:
    locations = None
    OPENAQ_COUNTRY_ID = OPENAQ_PM25_PARAMETER_ID = None
    print("RESUME - skipping OpenAQ discovery.")
else:
    OPENAQ_COUNTRY_ID = openaq_country_id(CONFIG["country"])
    OPENAQ_PM25_PARAMETER_ID = openaq_parameter_id("pm25")
    print(f"Resolved {CONFIG['country']} -> countries_id={OPENAQ_COUNTRY_ID}, "
      f"pm25 -> parameters_id={OPENAQ_PM25_PARAMETER_ID}")


def discover_india_pm25_locations():
    params = {
        "countries_id": OPENAQ_COUNTRY_ID,
        "parameters_id": OPENAQ_PM25_PARAMETER_ID,
        "limit": 1000,
    }
    return pd.json_normalize(openaq_paginate("/locations", params))


    locations = discover_india_pm25_locations()
    print("Discovered rows:", len(locations))

# Guard the geography rather than trusting the id: a wrong country id is
# otherwise invisible until someone reads a map.
    _codes = sorted(locations["country.code"].dropna().unique())
assert _codes == [CONFIG["country"]], f"Expected only {CONFIG['country']}, got {_codes}"
    _bb = CONFIG["india_bbox"]
    _lat, _lon = locations["coordinates.latitude"], locations["coordinates.longitude"]
    _outside = ((_lat < _bb["min_lat"]) | (_lat > _bb["max_lat"]) |
            (_lon < _bb["min_lon"]) | (_lon > _bb["max_lon"])).sum()
    print(f"Country codes: {_codes} | outside India bbox: {_outside}")
    print(f"lat {_lat.min():.2f}..{_lat.max():.2f} | lon {_lon.min():.2f}..{_lon.max():.2f}")
    display(locations.head())


### Station manifest quality gate

Before proceeding, inspect:

- number of unique locations
- number of unique sensors
- geographic extent
- state/city distribution
- expected historical coverage
- duplicate coordinates
- stations with very sparse history

Do **not** silently drop weak stations after model training. Record the exclusion reason in the manifest.

In [7]:
# ============================================================================
# STATION MANIFEST - resolve the LIVE pm25 sensor and rank by real coverage
# ============================================================================
# The manifest is the contract for acquisition: one row per station, carrying
# the sensor_id to fetch and the date range that sensor actually holds.
#
# Two-stage selection, because station-level metadata cannot be trusted:
#   Stage 1  cheap prefilter on the /locations payload (inside India, has a
#            pm25 sensor, reported recently).
#   Stage 2  authoritative per-sensor date ranges via /locations/{id}/sensors,
#            then rank by how much of the requested window each sensor covers.
#
# Stage 2 costs one request per candidate, which is why max_candidates is
# bounded. Sampling showed ~73% of Indian stations carry a usable sensor, so
# 400 candidates comfortably yields 150 stations.
# ============================================================================
def first_existing(df, candidates, default=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series(default, index=df.index)


# ---- Stage 1: cheap prefilter -------------------------------------------
cand = pd.DataFrame({
    "location_id": first_existing(locations, ["id", "locationsId"]),
    "station_name": first_existing(locations, ["name"]),
    "lat": pd.to_numeric(first_existing(locations, ["coordinates.latitude", "latitude"]), errors="coerce"),
    "lon": pd.to_numeric(first_existing(locations, ["coordinates.longitude", "longitude"]), errors="coerce"),
    "timezone": first_existing(locations, ["timezone"]),
    "provider": first_existing(locations, ["provider.name"]),
    "loc_last": pd.to_datetime(first_existing(locations, ["datetimeLast.utc"]), utc=True, errors="coerce"),
})
b = CONFIG["india_bbox"]
cand = cand[cand.lat.between(b["min_lat"], b["max_lat"]) &
            cand.lon.between(b["min_lon"], b["max_lon"])].drop_duplicates("location_id")
# Rank by recency so the sensor-resolution budget is spent on live stations.
cand = cand.sort_values("loc_last", ascending=False, na_position="last")
cand = cand.head(CONFIG["max_candidates"])
print(f"Stage 1: {len(cand)} candidate locations inside the India bbox")

# ---- Stage 2: authoritative per-sensor ranges ---------------------------
req_start = None if CONFIG["start"] == "auto" else pd.Timestamp(CONFIG["start"])
req_end = pd.Timestamp.now(tz="UTC") if CONFIG["end"] == "auto" else pd.Timestamp(CONFIG["end"])

rows = []
for rec in tqdm(cand.to_dict (orient="records"), desc="resolving pm25 sensors"):
    try:
        sensors = location_pm25_sensors(rec["location_id"])
    except Exception as e:
        print(f"  sensor lookup failed {rec['location_id']}: {e}")
        continue
    if not sensors:
        continue
    # Keep the sensor with the longest overlap with the requested window.
    def overlap(s):
        lo = s["sensor_first"] if req_start is None else max(s["sensor_first"], req_start)
        hi = min(s["sensor_last"], req_end)
        return max(pd.Timedelta(0), hi - lo)
    best = max(sensors, key=overlap)
    rows.append({**rec, **best, "overlap_hours": overlap(best).total_seconds() / 3600.0})
    time.sleep(CONFIG["throttle_s"])

manifest = pd.DataFrame(rows)
manifest = manifest[manifest.overlap_hours > 0].copy()
print(f"Stage 2: {len(manifest)} locations with a pm25 sensor holding data in the window")

# ---- Resolve the acquisition window from what the data actually holds ----
# The 25th percentile of sensor start dates keeps ~75% of stations fully
# covered rather than stretching the window back for a handful of outliers.
if CONFIG["start"] == "auto":
    WINDOW_START = manifest["sensor_first"].quantile(0.25).ceil("h")
else:
    WINDOW_START = pd.Timestamp(CONFIG["start"])
WINDOW_END = req_end.floor("h")

window_hours = (WINDOW_END - WINDOW_START).total_seconds() / 3600.0
manifest["window_coverage"] = (
    manifest.apply(lambda r: max(pd.Timedelta(0),
                                 min(r.sensor_last, WINDOW_END) - max(r.sensor_first, WINDOW_START)
                                 ).total_seconds() / 3600.0, axis=1) / window_hours
)
manifest = manifest[manifest.window_coverage >= CONFIG["min_coverage"]]
manifest = manifest.sort_values("window_coverage", ascending=False).head(CONFIG["max_stations"])
manifest["country"] = CONFIG["country"]
manifest["manifest_version"] = CONFIG["schema_version"]
manifest["selection_reason"] = "live_pm25_sensor_window_overlap"
manifest = manifest.reset_index(drop=True)

print(f"\nAcquisition window: {WINDOW_START} -> {WINDOW_END}  ({window_hours/24:.0f} days)")
print(f"Selected stations : {len(manifest)} (cap {CONFIG['max_stations']})")
print(f"Median coverage   : {manifest.window_coverage.median():.1%}")
print(f"Providers         : {manifest.provider.value_counts().head(5).to_dict()}")
display(manifest[["location_id","sensor_id","station_name","lat","lon",
                  "sensor_first","sensor_last","window_coverage"]].head(10))


Stage 1: 400 candidate locations inside the India bbox


resolving pm25 sensors:  31%|██████████████████████████████████████████████▏                                                                                                      | 124/400 [03:08<37:53,  8.24s/it]

  sensor lookup failed 42240: 500 from /locations/42240/sensors


resolving pm25 sensors:  54%|████████████████████████████████████████████████████████████████████████████████▍                                                                    | 216/400 [05:24<24:42,  8.06s/it]

  sensor lookup failed 6404322: 429 from /locations/6404322/sensors


...

resolving pm25 sensors:  89%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                | 357/400 [13:59<05:59,  8.36s/it]

  sensor lookup failed 7880: 500 from /locations/7880/sensors


resolving pm25 sensors: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [14:46<00:00,  2.22s/it]

Stage 2: 384 locations with a pm25 sensor holding data in the window

Acquisition window: 2025-02-18 21:00:00+00:00 -> 2026-09-09 10:00:00+00:00  (568 days)
Selected stations : 150 (cap 150)
Median coverage   : 99.9%
Providers         : {'CPCB': 144, 'AirNow': 3, 'N/A': 2, 'Clarity': 1}


,location_id,sensor_id,station_name,lat,lon,sensor_first,sensor_last,window_coverage
0,8558,5077812,Chennai,13.087840,80.278475,2022-10-03 20:00:00+00:00,2026-09-09 09:30:00+00:00,0.999963
1,8039,23320,Mumbai,19.072830,72.882610,2016-11-09 19:00:00+00:00,2026-09-09 09:30:00+00:00,0.999963
2,8118,23534,New Delhi,28.635760,77.224450,2016-11-09 19:00:00+00:00,2026-09-09 09:30:00+00:00,0.999963
3,3024966,10326358,NASA GSFC Rutgers Calib. N8,28.317330,76.916030,2024-08-20 06:43:12+00:00,2026-09-09 09:00:29+00:00,0.999927
4,3409506,12238531,"CDA Area, Cuttack - OSPCB",20.488910,85.847680,2025-02-18 20:15:00+00:00,2026-09-08 14:30:00+00:00,0.998568
5,3409505,12238522,"Lingraj Mandir, Bhubaneswar - OSPCB",20.240790,85.836784,2025-02-18 20:15:00+00:00,2026-09-08 14:30:00+00:00,0.998568
6,3409504,12238513,"Hakimapada, Angul - OSPCB",20.832874,85.104082,2025-02-18 20:15:00+00:00,2026-09-08 14:30:00+00:00,0.998568
7,3409503,12238504,"Forest Office, Barbil - OSPCB",22.116605,85.394555,2025-02-18 20:15:00+00:00,2026-09-08 14:30:00+00:00,0.998568
8,3409508,12238549,"Patia, Bhubaneswar - OSPCB",20.346520,85.816300,2025-02-18 20:15:00+00:00,2026-09-08 14:30:00+00:00,0.998568
9,3409520,12238649,"PCBL Residential Complex, Durgapur - WBPCB",23.508764,87.354440,2025-02-18 20:15:00+00:00,2026-09-08 14:30:00+00:00,0.998568


## 3. Fetch historical hourly PM2.5

Use OpenAQ's **hourly resource** rather than repeatedly downloading raw measurements where hourly data are sufficient.

OpenAQ documents `/v3/sensors/{sensor_id}/hours` as the hourly mean resource, with coverage metadata. Time ranges should be bounded because large queries are expensive. citeturn0search0turn0search5

Fetch in bounded chunks, e.g. one month at a time.

Never overwrite existing raw files without versioning.

In [10]:
# ============================================================================
# HISTORICAL PM2.5 ACQUISITION - the loop that was missing
# ============================================================================
# The previous version defined fetch_sensor_hours() and never called it, which
# is why the notebook produced no dataset. It also passed date_from/date_to,
# which OpenAQ v3 SILENTLY IGNORES - verified against the live API, a request
# for Jan-2023 returned 2016 rows, identical to sending no filter at all. The
# correct parameters are datetime_from / datetime_to.
#
# Requests are chunked by month so a single failure costs one month, not a
# whole station, and each completed station is cached to Parquet so an
# interrupted run resumes instead of restarting.
# ============================================================================
def month_ranges(start, end):
    """Inclusive-exclusive month boundaries covering [start, end)."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    cur = start
    while cur < end:
        nxt = min((cur + pd.offsets.MonthBegin(1)).normalize(), end)
        if nxt <= cur:
            nxt = end
        yield cur, nxt
        cur = nxt


_ISO = "%Y-%m-%dT%H:%M:%SZ"


def fetch_sensor_hours(sensor_id, start, end):
    """Hourly aggregates for one sensor over one window.

    NOTE the parameter names: datetime_from / datetime_to. date_from / date_to
    are accepted by the API and then ignored, returning the sensor's earliest
    data regardless of the window requested.
    """
    return openaq_paginate(
        f"/sensors/{int(sensor_id)}/hours",
        {
            "datetime_from": pd.Timestamp(start).tz_convert("UTC").strftime(_ISO),
            "datetime_to": pd.Timestamp(end).tz_convert("UTC").strftime(_ISO),
            "limit": 1000,
        },
    )


def acquire_station(rec, start, end, use_cache=True):
    """All hourly rows for one station, cached to Parquet for resume."""
    cache = STATION_CACHE / f"sensor_{int(rec['sensor_id'])}.parquet"
    if use_cache and cache.exists():
        return pd.read_parquet(cache), True

    frames = []
    for lo, hi in month_ranges(start, end):
        # Skip months the sensor provably cannot cover - saves ~30% of calls.
        if hi <= rec["sensor_first"] or lo >= rec["sensor_last"]:
            continue
        try:
            raw = pd.json_normalize(fetch_sensor_hours(rec["sensor_id"], lo, hi))
        except Exception as e:
            print(f"  {rec['station_name'][:28]} {lo:%Y-%m}: {e}")
            continue
        if not raw.empty:
            frames.append(raw)
        time.sleep(CONFIG["throttle_s"])

    raw_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    df = normalize_pm25(
        raw_all,
        location_id=rec["location_id"], sensor_id=rec["sensor_id"],
        station_name=rec["station_name"], lat=rec["lat"], lon=rec["lon"],
    )
    df.to_parquet(cache, index=False)
    return df, False


t0 = time.time()
parts, cached_n = [], 0
for rec in tqdm(manifest.to_dict(orient="records"), desc="stations"):
    df, was_cached = acquire_station(rec, WINDOW_START, WINDOW_END)
    cached_n += int(was_cached)
    if not df.empty:
        parts.append(df)

pm25_all = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print(f"\nAcquired {len(pm25_all):,} hourly rows from {len(parts)} stations "
      f"in {time.time()-t0:.0f}s ({cached_n} served from cache)")
if pm25_all.empty:
    raise RuntimeError("No PM2.5 history returned - check the window and manifest.")
print(f"Time span : {pm25_all.timestamp_utc.min()} -> {pm25_all.timestamp_utc.max()}")
print(f"PM2.5     : {pm25_all.pm25.min():.1f} .. {pm25_all.pm25.max():.1f} ug/m3 "
      f"(median {pm25_all.pm25.median():.1f})")
display(pm25_all.head())


stations: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 150/150 [56:50<00:00, 22.74s/it]


Acquired 1,706,077 hourly rows from 150 stations in 3411s (65 served from cache)
Time span : 2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00
PM2.5     : 0.0 .. 418000.0 ug/m3 (median 28.3)


,timestamp_utc,location_id,sensor_id,station_name,lat,lon,pm25,pm25_unit,source,pm25_extreme_flag,pm25_severe_flag
0,2025-02-18 20:30:00+00:00,8558,5077812,Chennai,13.08784,80.278475,51.0,ug/m3,OpenAQ,False,False
1,2025-02-18 21:30:00+00:00,8558,5077812,Chennai,13.08784,80.278475,49.0,ug/m3,OpenAQ,False,False
2,2025-02-18 22:30:00+00:00,8558,5077812,Chennai,13.08784,80.278475,49.0,ug/m3,OpenAQ,False,False
3,2025-02-18 23:30:00+00:00,8558,5077812,Chennai,13.08784,80.278475,65.0,ug/m3,OpenAQ,False,False
4,2025-02-19 00:30:00+00:00,8558,5077812,Chennai,13.08784,80.278475,94.0,ug/m3,OpenAQ,False,True


## 4. Normalize and quality-control PM2.5

Rules:

- normalize timestamps to UTC
- preserve original timestamp in `source_timestamp`
- convert units to µg/m³
- remove impossible negative values
- do not clip extreme values merely because they are rare
- flag extreme values instead
- collapse exact duplicate `(sensor_id, timestamp)` rows
- retain station gaps as missingness information
- calculate station coverage against the expected hourly grid

**Do not remove high PM2.5 simply because it hurts model performance.** Extreme observations are exactly what the next phases need.

In [9]:
# ============================================================================
# NORMALIZE PM2.5 into the canonical row contract
# ============================================================================
# Called by acquire_station() for every station. Responsibilities:
#   - pick whichever timestamp field the payload actually carries
#   - coerce values, drop nulls and physically impossible negatives
#   - flag severe (>=90) and extreme (>=150) rows so later phases can stratify
#   - de-duplicate on (sensor_id, timestamp), keeping the latest revision
#
# period.datetimeFrom is preferred over datetimeTo: an hourly aggregate is
# labelled by the START of its hour everywhere else in AeroPulse, and mixing
# the two would shift every observation by one hour relative to weather.
# ============================================================================

def normalize_pm25(raw, location_id, sensor_id, station_name, lat, lon):
    if raw.empty:
        return pd.DataFrame(columns=[
            "timestamp_utc","location_id","sensor_id","station_name","lat","lon","pm25"
        ])

    out = pd.DataFrame()
    out["timestamp_utc"] = pd.to_datetime(
        first_existing(raw, ["period.datetimeFrom.utc", "period.datetimeTo.utc", "datetime.utc"]),
        utc=True, errors="coerce"
    )
    out["pm25"] = pd.to_numeric(first_existing(raw, ["value"]), errors="coerce")
    out["location_id"] = location_id
    out["sensor_id"] = sensor_id
    out["station_name"] = station_name
    out["lat"] = lat
    out["lon"] = lon
    out["source"] = "OpenAQ"
    out["pm25_unit"] = "ug/m3"

    out = out.dropna(subset=["timestamp_utc","pm25"])
    out = out[out["pm25"] >= 0]
    out["pm25_extreme_flag"] = out["pm25"] >= 150
    out["pm25_severe_flag"] = out["pm25"] >= 90
    out = out.drop_duplicates(["sensor_id","timestamp_utc"], keep="last")
    return out

print("normalize_pm25 ready - called per station by the acquisition loop.")


normalize_pm25 ready - called per station by the acquisition loop.


## 5. Persist raw and normalized PM2.5

Recommended layout:

```text
data/pm25/
  raw/
    openaq/
      year=2023/month=01/...
  processed/
    base/
      pm25_hourly.parquet
      station_manifest.parquet
      data_quality_report.json
```

The raw layer is immutable.

The processed layer is reproducible from raw data + notebook version.

In [11]:
# ============================================================================
# PERSIST PM2.5 + MANIFEST  (these writes were commented out before)
# ============================================================================
# Applying the per-station row/coverage gates here rather than during
# acquisition is deliberate: weak stations stay in the manifest with an
# explicit exclusion reason, so the drop is auditable instead of invisible.
# ============================================================================
def dataset_fingerprint(df):
    """Order-independent content hash, for provenance in the manifest."""
    payload = pd.util.hash_pandas_object(
        df.sort_index(axis=1).sort_values(list(df.columns)).reset_index(drop=True),
        index=False
    ).values.tobytes()
    return hashlib.sha256(payload).hexdigest()


station_rows = pm25_all.groupby("sensor_id").size().rename("rows")
expected = (WINDOW_END - WINDOW_START).total_seconds() / 3600.0
station_cov = (station_rows / expected).rename("observed_coverage")

manifest = manifest.merge(station_rows, on="sensor_id", how="left") \
                   .merge(station_cov, on="sensor_id", how="left")
manifest["rows"] = manifest["rows"].fillna(0).astype(int)
manifest["observed_coverage"] = manifest["observed_coverage"].fillna(0.0)
manifest["excluded"] = manifest["rows"] < CONFIG["min_station_rows"]
manifest["exclusion_reason"] = np.where(
    manifest["excluded"], f"rows < min_station_rows({CONFIG['min_station_rows']})", ""
)

keep = set(manifest.loc[~manifest.excluded, "sensor_id"])
pm25_all = pm25_all[pm25_all.sensor_id.isin(keep)].copy()

print(f"Stations kept    : {(~manifest.excluded).sum()} / {len(manifest)}")
print(f"Stations excluded: {manifest.excluded.sum()} (below {CONFIG['min_station_rows']} rows)")
print(f"Rows after gate  : {len(pm25_all):,}")
print(f"Observed coverage: median {manifest.loc[~manifest.excluded,'observed_coverage'].median():.1%}")

pm25_all.to_parquet(BASE_DIR / "pm25_hourly.parquet", index=False)
manifest.to_parquet(BASE_DIR / "station_manifest.parquet", index=False)
print("\nWrote:", BASE_DIR / "pm25_hourly.parquet")
print("Wrote:", BASE_DIR / "station_manifest.parquet")


Stations kept    : 149 / 150
Stations excluded: 1 (below 1000 rows)
Rows after gate  : 1,705,288
Observed coverage: median 86.3%

Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/pm25_hourly.parquet
Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/station_manifest.parquet


## 6. Weather source adapter

### Recommended architecture

Keep this interface stable:

```text
get_weather_hourly(lat, lon, start, end)
        ↓
canonical weather schema
```

The implementation can initially use the current weather provider.

Later it can use IMD.

**Downstream feature engineering does not change if the canonical schema stays stable.**

IMD currently publishes long historical gridded rainfall and temperature products. For example, IMD has 0.25° daily gridded rainfall data and 1° daily temperature archives. citeturn1search1turn1search0

However, those products are not a drop-in replacement for the hourly weather variables currently used by the notebook. Therefore, use an adapter rather than changing the model schema.

In [12]:
# ============================================================================
# WEATHER SOURCE ADAPTER - Open-Meteo ERA5 archive, one call per station
# ============================================================================
# Weather drives dispersion, so these are the physical features.
# boundary_layer_height matters most: a shallow layer traps pollution near the
# surface and is the usual cause of winter PM2.5 spikes.
#
# The adapter shape (normalize_weather_source) is what keeps this swappable -
# an IMD connector can replace Open-Meteo later without touching any
# downstream notebook, provided it emits WEATHER_COLUMNS.
# ============================================================================
OPEN_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "surface_pressure",
    "precipitation", "wind_speed_10m", "wind_direction_10m",
    "boundary_layer_height",
]
WEATHER_COLUMNS = ["timestamp_utc", "lat", "lon"] + WEATHER_VARS + ["wind_u", "wind_v", "weather_source"]


WEATHER_CACHE = RAW_DIR / "weather"
WEATHER_CACHE.mkdir(parents=True, exist_ok=True)


def _http_json(url, params, retries=6, backoff=2.0, timeout=120):
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(str(r.status_code), response=r)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            # Includes DNS failures (NameResolutionError). A transient resolver
            # outage wiped 144 of 149 stations on the first full run, so the
            # backoff here is deliberately long.
            last = e
            if attempt == retries - 1:
                break
            time.sleep(backoff * (2 ** attempt))
    raise last


def fetch_weather_archive(lat, lon, start, end):
    payload = _http_json(OPEN_METEO_ARCHIVE, {
        "latitude": lat, "longitude": lon,
        "start_date": pd.Timestamp(start).date().isoformat(),
        "end_date": pd.Timestamp(end).date().isoformat(),
        "hourly": ",".join(WEATHER_VARS),
        "timezone": "UTC",
    })
    df = pd.DataFrame(payload.get("hourly", {}))
    if "time" in df:
        df["timestamp_utc"] = pd.to_datetime(df.pop("time"), utc=True)
    return df


def normalize_weather_source(df, lat, lon, source_name="open-meteo-era5"):
    """Map any provider into AeroPulse's canonical weather contract."""
    out = df.copy()
    out["lat"], out["lon"] = lat, lon
    out["weather_source"] = source_name
    # Wind direction is circular: 359 deg and 1 deg are adjacent but numerically
    # far apart, which trees split badly. u/v components make it continuous.
    wd = np.deg2rad(pd.to_numeric(out.get("wind_direction_10m"), errors="coerce"))
    ws = pd.to_numeric(out.get("wind_speed_10m"), errors="coerce")
    out["wind_u"] = -np.sin(wd) * ws
    out["wind_v"] = -np.cos(wd) * ws
    for c in WEATHER_COLUMNS:
        if c not in out.columns:
            out[c] = np.nan
    return out[WEATHER_COLUMNS]


stations = manifest.loc[~manifest.excluded, ["lat", "lon"]].drop_duplicates()
t0, wparts, failed = time.time(), [], []
for rec in tqdm(stations.to_dict(orient="records"), desc="weather"):
    # Cache per coordinate so a network blip does not cost the whole fetch,
    # and so re-running the notebook is nearly free.
    cache = WEATHER_CACHE / f"w_{rec['lat']:.4f}_{rec['lon']:.4f}.parquet"
    if cache.exists():
        wparts.append(pd.read_parquet(cache))
        continue
    try:
        raw = fetch_weather_archive(rec["lat"], rec["lon"], WINDOW_START, WINDOW_END)
        if not raw.empty:
            norm = normalize_weather_source(raw, rec["lat"], rec["lon"])
            norm.to_parquet(cache, index=False)
            wparts.append(norm)
    except Exception as e:
        failed.append((rec["lat"], rec["lon"]))
        print(f"  weather failed at {rec['lat']:.3f},{rec['lon']:.3f}: {type(e).__name__}")

weather = pd.concat(wparts, ignore_index=True) if wparts else pd.DataFrame(columns=WEATHER_COLUMNS)
# float32 halves memory on a multi-million-row frame with no meaningful
# precision loss for meteorological quantities.
for c in WEATHER_VARS + ["wind_u", "wind_v"]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce").astype("float32")

weather.to_parquet(BASE_DIR / "weather_hourly.parquet", index=False)
print(f"\nWeather rows: {len(weather):,} for {len(stations)-len(failed)}/{len(stations)} "
      f"stations in {time.time()-t0:.0f}s")
if failed:
    print(f"FAILED for {len(failed)} stations - rerun this cell to retry only those "
          f"(successful ones are cached).")
print("Wrote:", BASE_DIR / "weather_hourly.parquet")
display(weather.head())


weather:   4%|██████▌                                                                                                                                                             | 6/149 [02:19<1:49:11, 45.82s/it]

  weather failed at 20.833,85.104: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=20.832874&longitude=85.104082&start_date=2025-02-18&end_date=2026-09-09&hourly=temperature_2m%2Crelative_humidity_2m%2Csurface_pressure%2Cprecipitation%2Cwind_speed_10m%2Cwind_direction_10m%2Cboundary_layer_height&timezone=UTC (Caused by NameResolutionError("HTTPSConnection(host='archive-api.open-meteo.com', port=443): Failed to resolve 'archive-api.open-meteo.com' ([Errno 8] nodename nor servname provided, or not known)"))


weather:   5%|███████▋                                                                                                                                                            | 7/149 [02:30<1:21:07, 34.28s/it]

  weather failed at 22.117,85.395: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=22.1166054&longitude=85.3945546&start_date=2025-02-18&end_date=2026-09-09&hourly=temperature_2m%2Crelative_humidity_2m%2Csurface_pressure%2Cprecipitation%2Cwind_speed_10m%2Cwind_direction_10m%2Cboundary_layer_height&timezone=UTC (Caused by NameResolutionError("HTTPSConnection(host='archive-api.open-meteo.com', port=443): Failed to resolve 'archive-api.open-meteo.com' ([Errno 8] nodename nor servname provided, or not known)"))


weather:   5%|████████▊                                                                                                                                                           | 8/149 [02:40<1:02:47, 26.72s/it]

...

weather:  99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉ | 148/149 [27:16<00:10, 10.54s/it]

  weather failed at 28.841,78.698: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=28.840739000000003&longitude=78.69753100000001&start_date=2025-02-18&end_date=2026-09-09&hourly=temperature_2m%2Crelative_humidity_2m%2Csurface_pressure%2Cprecipitation%2Cwind_speed_10m%2Cwind_direction_10m%2Cboundary_layer_height&timezone=UTC (Caused by NameResolutionError("HTTPSConnection(host='archive-api.open-meteo.com', port=443): Failed to resolve 'archive-api.open-meteo.com' ([Errno 8] nodename nor servname provided, or not known)"))


weather: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [27:26<00:00, 11.05s/it]

  weather failed at 23.164,72.678: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=23.163798&longitude=72.677768&start_date=2025-02-18&end_date=2026-09-09&hourly=temperature_2m%2Crelative_humidity_2m%2Csurface_pressure%2Cprecipitation%2Cwind_speed_10m%2Cwind_direction_10m%2Cboundary_layer_height&timezone=UTC (Caused by NameResolutionError("HTTPSConnection(host='archive-api.open-meteo.com', port=443): Failed to resolve 'archive-api.open-meteo.com' ([Errno 8] nodename nor servname provided, or not known)"))

Weather rows: 68,280 for 149 stations in 1647s
Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/weather_hourly.parquet


,timestamp_utc,lat,lon,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,boundary_layer_height,wind_u,wind_v,weather_source
0,2025-02-18 00:00:00+00:00,13.08784,80.278475,22.799999,94.0,1011.000000,0.0,4.0,265.0,50.0,3.984779,0.348623,open-meteo-era5
1,2025-02-18 01:00:00+00:00,13.08784,80.278475,22.700001,96.0,1011.700012,0.0,5.5,283.0,75.0,5.359035,-1.237231,open-meteo-era5
2,2025-02-18 02:00:00+00:00,13.08784,80.278475,22.900000,96.0,1012.599976,0.0,5.2,274.0,85.0,5.187333,-0.362734,open-meteo-era5
3,2025-02-18 03:00:00+00:00,13.08784,80.278475,25.500000,82.0,1013.700012,0.0,3.5,282.0,220.0,3.423517,-0.727691,open-meteo-era5
4,2025-02-18 04:00:00+00:00,13.08784,80.278475,28.100000,68.0,1014.500000,0.0,4.8,149.0,845.0,-2.472183,4.114403,open-meteo-era5


## 7. Fire source adapter

FIRMS should remain an independent source.

Store raw fire detections once and derive spatial features later.

Do not calculate `fire_count_20km` only at model-training time. Preserve the raw detections so later notebooks can test:

- 5 km
- 10 km
- 20 km
- 50 km
- 100 km
- upwind-only
- FRP weighted
- time-decayed fire impact

This prevents another expensive re-download.

In [13]:
# ============================================================================
# FIRE SOURCE ADAPTER - NASA FIRMS, multi-radius evidence
# ============================================================================
# In this region stubble burning is the seasonal driver of severe PM2.5, so
# this is the feature expected to explain extreme events.
#
# Two API constraints shape this code:
#   1. A single /area query is capped at 5 days ("Invalid day range.
#      Expects [1..5]"), so a long window must be walked in chunks.
#   2. NRT products only retain a recent rolling window; older dates need the
#      science-quality (SP) product. firms_source_for() picks per date, so a
#      multi-month window silently returning nothing is not possible.
# ============================================================================
FIRMS_BASE = "https://firms.modaps.eosdis.nasa.gov/api"
FIRMS_MAX_DAY_RANGE = 5
FIRMS_SP_SOURCE, FIRMS_NRT_SOURCE = "VIIRS_NOAA20_SP", "VIIRS_NOAA20_NRT"


def firms_availability():
    """Per-product date coverage, so source choice is data-driven not guessed."""
    key = require_env("FIRMS_MAP_KEY")
    r = requests.get(f"{FIRMS_BASE}/data_availability/csv/{key}/ALL", timeout=120)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    return {row.data_id: (pd.Timestamp(row.min_date), pd.Timestamp(row.max_date))
            for row in df.itertuples()}


FIRMS_AVAIL = firms_availability()
print("FIRMS availability:", {k: (str(v[0].date()), str(v[1].date()))
                              for k, v in FIRMS_AVAIL.items() if "VIIRS_NOAA20" in k})


def firms_source_for(day):
    """SP for archival dates, NRT for recent ones."""
    day = pd.Timestamp(day).tz_localize(None).normalize()
    for src in (FIRMS_SP_SOURCE, FIRMS_NRT_SOURCE):
        lo, hi = FIRMS_AVAIL.get(src, (None, None))
        if lo is not None and lo <= day <= hi:
            return src
    return None


def firms_area_csv(source, bbox, days, start_date):
    key = require_env("FIRMS_MAP_KEY")
    area = f'{bbox["min_lon"]},{bbox["min_lat"]},{bbox["max_lon"]},{bbox["max_lat"]}'
    url = (f"{FIRMS_BASE}/area/csv/{key}/{source}/{area}/"
           f"{int(min(days, FIRMS_MAX_DAY_RANGE))}/{pd.Timestamp(start_date).date().isoformat()}")
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    text = r.text.strip()
    # FIRMS signals some errors as HTTP 200 with a plain-text body, not CSV.
    if not text or not text.lower().startswith("latitude"):
        raise RuntimeError(f"non-CSV body: {text[:120]}")
    return pd.read_csv(io.StringIO(text))


FIRE_CACHE = RAW_DIR / "fire"
FIRE_CACHE.mkdir(parents=True, exist_ok=True)


def firms_area_range(bbox, start, end):
    """Walk the window in 5-day chunks, caching each so reruns are cheap."""
    frames = []
    cursor = pd.Timestamp(start).tz_localize(None).normalize()
    stop = pd.Timestamp(end).tz_localize(None).normalize()
    total = max(1, int((stop - cursor).days / FIRMS_MAX_DAY_RANGE) + 1)
    with tqdm(total=total, desc="FIRMS chunks") as bar:
        while cursor <= stop:
            src = firms_source_for(cursor)
            chunk_cache = FIRE_CACHE / f"{src}_{cursor.date()}.parquet" if src else None
            if src is None:
                print(f"  no FIRMS product covers {cursor.date()}, skipped")
            elif chunk_cache.exists():
                frames.append(pd.read_parquet(chunk_cache))
            else:
                try:
                    chunk = firms_area_csv(src, bbox, FIRMS_MAX_DAY_RANGE, cursor)
                    chunk.to_parquet(chunk_cache, index=False)
                    frames.append(chunk)
                except Exception as e:
                    print(f"  FIRMS {cursor.date()} ({src}): {type(e).__name__}")
            cursor += pd.Timedelta(days=FIRMS_MAX_DAY_RANGE)
            bar.update(1)
    return pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()


FIRE_COLUMNS = ["timestamp_utc", "fire_lat", "fire_lon", "frp", "confidence", "fire_source"]


def normalize_fire_source(df, source_name="FIRMS/VIIRS"):
    if df.empty:
        return pd.DataFrame(columns=FIRE_COLUMNS)
    out = pd.DataFrame({
        "timestamp_utc": pd.to_datetime(
            df["acq_date"].astype(str) + " " + df["acq_time"].astype(str).str.zfill(4),
            format="%Y-%m-%d %H%M", utc=True, errors="coerce"),
        "fire_lat": pd.to_numeric(df["latitude"], errors="coerce"),
        "fire_lon": pd.to_numeric(df["longitude"], errors="coerce"),
        "frp": pd.to_numeric(df.get("frp"), errors="coerce").fillna(0.0),
        "confidence": df.get("confidence"),
    })
    out["fire_source"] = source_name
    return out.dropna(subset=["timestamp_utc", "fire_lat", "fire_lon"])[FIRE_COLUMNS]


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    lat1, lon1 = np.asarray(lat1, float), np.asarray(lon1, float)
    lat2, lon2 = np.asarray(lat2, float), np.asarray(lon2, float)
    dphi, dl = np.deg2rad(lat2 - lat1), np.deg2rad(lon2 - lon1)
    a = (np.sin(dphi / 2) ** 2 +
         np.cos(np.deg2rad(lat1)) * np.cos(np.deg2rad(lat2)) * np.sin(dl / 2) ** 2)
    return 2 * r * np.arcsin(np.sqrt(a))


def add_fire_features(df, fire, radii_km, window_hours):
    """Fire count / FRP within each radius in the window BEFORE each row.

    Strictly backward-looking, so no future fire can leak into a feature.
    Per station: one distance pass over all fires, then prefix-sum lookups per
    timestamp - not a re-filter of the fire frame for every row.
    """
    out = df.copy()
    for rk in radii_km:
        out[f"fire_count_{int(rk)}km"] = 0.0
        out[f"fire_frp_{int(rk)}km"] = 0.0
    out["fire_min_distance_km"] = np.nan
    if fire.empty:
        return out

    def naive(s):
        # tz-aware Series -> object array under to_numpy(); searchsorted needs
        # a real datetime64 dtype.
        return s.dt.tz_convert("UTC").dt.tz_localize(None).to_numpy("datetime64[ns]")

    flat, flon = fire["fire_lat"].to_numpy(float), fire["fire_lon"].to_numpy(float)
    fire_sorted = fire.sort_values("timestamp_utc")
    ft_all = naive(fire_sorted["timestamp_utc"])
    for (lat, lon), idx in tqdm(out.groupby(["lat", "lon"]).groups.items(), desc="fire features"):
        dist = haversine_km(float(lat), float(lon), flat, flon)
        ts = naive(out.loc[idx, "timestamp_utc"])
        lo_t = ts - np.timedelta64(window_hours, "h")
        for rk in radii_km:
            mask = dist <= rk
            if not mask.any():
                continue
            near = fire.loc[mask, ["timestamp_utc", "frp"]].sort_values("timestamp_utc")
            ft = naive(near["timestamp_utc"])
            cum = np.concatenate([[0.0], near["frp"].to_numpy(float).cumsum()])
            hi = np.searchsorted(ft, ts, side="right")
            lo = np.searchsorted(ft, lo_t, side="right")
            out.loc[idx, f"fire_count_{int(rk)}km"] = (hi - lo).astype(float)
            out.loc[idx, f"fire_frp_{int(rk)}km"] = cum[hi] - cum[lo]
        # Nearest fire overall, regardless of radius.
        if len(dist):
            out.loc[idx, "fire_min_distance_km"] = float(dist.min())
    return out


t0 = time.time()
fire_raw = firms_area_range(CONFIG["india_bbox"], WINDOW_START, WINDOW_END)
fire = normalize_fire_source(fire_raw)
fire.to_parquet(BASE_DIR / "fire_detections.parquet", index=False)
print(f"\nFIRMS detections: {len(fire):,} in {time.time()-t0:.0f}s")
if not fire.empty:
    print(f"Span : {fire.timestamp_utc.min()} -> {fire.timestamp_utc.max()}")
    print(f"FRP  : {fire.frp.min():.2f} .. {fire.frp.max():.2f}")
    monthly = fire.set_index("timestamp_utc").resample("MS").size()
    print("\nDetections per month (burning season should dominate):")
    print(monthly.to_string())
print("Wrote:", BASE_DIR / "fire_detections.parquet")


FIRMS availability: {'VIIRS_NOAA20_NRT': ('2026-06-01', '2026-09-09'), 'VIIRS_NOAA20_SP': ('2018-04-01', '2026-05-31')}


FIRMS chunks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 114/114 [05:17<00:00,  2.79s/it]



FIRMS detections: 1,692,885 in 319s
Span : 2025-02-18 06:21:00+00:00 -> 2026-09-09 09:07:00+00:00
FRP  : 0.00 .. 1230.11

Detections per month (burning season should dominate):
timestamp_utc
2025-02-01 00:00:00+00:00     48072
2025-03-01 00:00:00+00:00    364906
2025-04-01 00:00:00+00:00    193239
2025-05-01 00:00:00+00:00     72957
2025-06-01 00:00:00+00:00     15677
2025-07-01 00:00:00+00:00      4303
2025-08-01 00:00:00+00:00      5224
2025-09-01 00:00:00+00:00      6874
2025-10-01 00:00:00+00:00     23603
2025-11-01 00:00:00+00:00     68903
2025-12-01 00:00:00+00:00     70395
2026-01-01 00:00:00+00:00     66187
2026-02-01 00:00:00+00:00    118405
2026-03-01 00:00:00+00:00    240461
2026-04-01 00:00:00+00:00    261104
2026-05-01 00:00:00+00:00     96876
2026-06-01 00:00:00+00:00     22068
2026-07-01 00:00:00+00:00      5892
2026-08-01 00:00:00+00:00      4592
2026-09-01 00:00:00+00:00      3147
Freq: MS
Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebo

## 8. Point-in-time-safe weather join

Use nearest historical weather observation with a tolerance.

For training, the join must never use future weather observations.

For forecasting, future weather must come from the forecast product, not historical reanalysis.

This distinction is essential:

```text
Training:
past weather → target

Inference:
observed weather + forecast weather → future target
```

In [14]:
# ============================================================================
# POINT-IN-TIME-SAFE JOIN - assemble the canonical base dataset
# ============================================================================
# An exact merge on timestamp silently fails here: OpenAQ stamps each station
# in its own local-offset grid (IST stations land on HH:30) while Open-Meteo is
# always HH:00, so an equality join drops every Indian station's weather
# without raising anything.
#
# merge_asof(direction="backward") takes the most recent weather hour AT OR
# BEFORE each observation. That tolerates the offset and - critically - never
# pulls a FUTURE weather hour into a feature, which would be leakage.
# ============================================================================
def join_weather_point_in_time(pm25, weather, tolerance="90min"):
    left = pm25.sort_values("timestamp_utc")
    right = weather.sort_values("timestamp_utc")
    return pd.merge_asof(
        left, right,
        on="timestamp_utc", by=["lat", "lon"],
        direction="backward", tolerance=pd.Timedelta(tolerance),
        suffixes=("", "_weather"),
    )


base = join_weather_point_in_time(pm25_all, weather)
base = add_fire_features(base, fire, CONFIG["fire_radii_km"], CONFIG["fire_window_hours"])

# Local time is what humans reason about ("the 9am rush hour"), so carry both.
base["timestamp_local"] = base["timestamp_utc"].dt.tz_convert("Asia/Kolkata")
base["source_quality"] = "openaq_hourly_aggregate"
base["schema_version"] = CONFIG["schema_version"]

wcov = base["temperature_2m"].notna().mean()
fcov = (base["fire_count_100km"] > 0).mean()
print(f"Base dataset      : {len(base):,} rows x {base.shape[1]} columns")
print(f"Weather coverage  : {wcov:.1%}")
print(f"Rows with fire<100km in prior 24h: {fcov:.1%}")
if wcov < 0.95:
    worst = (base.groupby("station_name")["temperature_2m"]
             .apply(lambda s: s.isna().mean()).sort_values(ascending=False).head(5))
    print("WARNING - stations with the most missing weather:\n", worst)
display(base.head())


fire features: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [00:05<00:00, 29.34it/s]

Base dataset      : 1,705,288 rows x 31 columns
Weather coverage  : 3.2%
Rows with fire<100km in prior 24h: 64.7%
WARNING - stations with the most missing weather:
 station_name
AIIMS, Raipur - CECB                          1.0
Ramdaspeth, Akola - MPCB                      1.0
Park Street Wakad, Pimpri Chinchwad - MPCB    1.0
Paryavaran Parisar, Bhopal - MPPCB            1.0
Patia, Bhubaneswar - OSPCB                    1.0
Name: temperature_2m, dtype: float64


,timestamp_utc,location_id,sensor_id,station_name,lat,lon,pm25,pm25_unit,source,pm25_extreme_flag,...,fire_count_20km,fire_frp_20km,fire_count_50km,fire_frp_50km,fire_count_100km,fire_frp_100km,fire_min_distance_km,timestamp_local,source_quality,schema_version
0,2025-02-18 20:30:00+00:00,8558,5077812,Chennai,13.087840,80.278475,51.0,ug/m3,OpenAQ,False,...,2.0,3.99,4.0,10.19,5.0,15.29,2.922751,2025-02-19 02:00:00+05:30,openaq_hourly_aggregate,1.0.0
1,2025-02-18 20:30:00+00:00,3409449,12238095,"Dnyaneshwar Nagar, Solapur - MPCB",17.633626,75.913250,69.8,ug/m3,OpenAQ,False,...,16.0,71.49,49.0,215.65,84.0,464.60,1.674441,2025-02-19 02:00:00+05:30,openaq_hourly_aggregate,1.0.0
2,2025-02-18 20:30:00+00:00,3409508,12238549,"Patia, Bhubaneswar - OSPCB",20.346520,85.816300,35.0,ug/m3,OpenAQ,False,...,0.0,0.00,1.0,2.46,15.0,87.21,4.191067,2025-02-19 02:00:00+05:30,openaq_hourly_aggregate,1.0.0
3,2025-02-18 20:30:00+00:00,3409450,12238104,"Karni Colony, Nagaur - RSPCB",27.213494,73.734444,28.8,ug/m3,OpenAQ,False,...,0.0,0.00,0.0,0.00,1.0,2.08,13.167740,2025-02-19 02:00:00+05:30,openaq_hourly_aggregate,1.0.0
4,2025-02-18 20:30:00+00:00,3409451,12238112,"Pandav Nagari, Nashik - MPCB",19.959135,73.778801,114.0,ug/m3,OpenAQ,False,...,0.0,0.00,3.0,12.54,8.0,35.02,0.653288,2025-02-19 02:00:00+05:30,openaq_hourly_aggregate,1.0.0


## 9. Dataset quality gates

The notebook must fail loudly when:

- India filter is violated
- target coverage is too low
- duplicate target rows remain
- weather join coverage <95%
- timestamps are not monotonic within station
- units are inconsistent
- a station has insufficient history
- all extreme observations disappear
- fire observations are unexpectedly all zero

Create a machine-readable quality report and a human-readable summary.

In [15]:
# ============================================================================
# DATASET QUALITY GATES - fail loudly here, not three notebooks downstream
# ============================================================================
# Every gate answers a question a later phase would otherwise discover the hard
# way. season_coverage is the one that matters most for AeroPulse: a model
# trained only on monsoon months cannot predict a burning-season episode, and
# that failure is invisible in aggregate metrics.
# ============================================================================
def india_air_season(ts):
    """Indian air-quality regimes, which do not match meteorological seasons."""
    m = ts.dt.month
    return np.select(
        [m.isin([12, 1, 2]), m.isin([3, 4, 5]), m.isin([6, 7, 8, 9])],
        ["winter", "pre_monsoon", "monsoon"],
        default="post_monsoon",
    )


base["season"] = india_air_season(base["timestamp_local"])


def quality_report(pm25, weather=None, fire=None):
    b = CONFIG["india_bbox"]
    rep = {
        "schema_version": CONFIG["schema_version"],
        "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "rows": int(len(pm25)),
        "stations": int(pm25.location_id.nunique()),
        "sensors": int(pm25.sensor_id.nunique()),
        "min_timestamp": str(pm25.timestamp_utc.min()),
        "max_timestamp": str(pm25.timestamp_utc.max()),
        "span_days": float((pm25.timestamp_utc.max() - pm25.timestamp_utc.min()).days),
        "missing_pm25": int(pm25.pm25.isna().sum()),
        "negative_pm25": int((pm25.pm25 < 0).sum()),
        "extreme_rows": int((pm25.pm25 >= 150).sum()),
        "severe_rows": int((pm25.pm25 >= 90).sum()),
        "duplicate_sensor_hour": int(pm25.duplicated(["sensor_id", "timestamp_utc"]).sum()),
        "india_bbox_violations": int(
            (~pm25.lat.between(b["min_lat"], b["max_lat"]) |
             ~pm25.lon.between(b["min_lon"], b["max_lon"])).sum()),
        "season_coverage": {k: int(v) for k, v in pm25.season.value_counts().items()},
        "weather_coverage": float(pm25["temperature_2m"].notna().mean()),
    }
    if weather is not None:
        rep["weather_rows"] = int(len(weather))
    if fire is not None:
        rep["fire_detections"] = int(len(fire))
    return rep


report = quality_report(base, weather, fire)

# Gates. A warning here is a reason to widen the window, not to proceed.
GATES = {
    "no duplicate sensor-hours": report["duplicate_sensor_hour"] == 0,
    "no negative PM2.5": report["negative_pm25"] == 0,
    "all inside India bbox": report["india_bbox_violations"] == 0,
    "weather coverage >= 95%": report["weather_coverage"] >= 0.95,
    "has extreme (>=150) rows": report["extreme_rows"] > 0,
    "covers >= 3 seasons": len(report["season_coverage"]) >= 3,
    "includes post_monsoon": "post_monsoon" in report["season_coverage"],
}
print(json.dumps(report, indent=2))
print("\nQUALITY GATES")
for name, ok in GATES.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
if not all(GATES.values()):
    print("\nWARNING: one or more gates failed. Phase 2 will run, but read the "
          "failures above before trusting anything trained on this dataset.")


{
  "schema_version": "1.0.0",
  "generated_utc": "2026-09-09T12:24:28.271764+00:00",
  "rows": 1705288,
  "stations": 149,
  "sensors": 149,
  "min_timestamp": "2025-02-18 20:30:00+00:00",
  "max_timestamp": "2026-09-09 07:30:00+00:00",
  "span_days": 567.0,
  "missing_pm25": 0,
  "negative_pm25": 0,
  "extreme_rows": 19093,
  "severe_rows": 89761,
  "duplicate_sensor_hour": 0,
  "india_bbox_violations": 0,
  "season_coverage": {
    "monsoon": 673073,
    "pre_monsoon": 562227,
    "winter": 280215,
    "post_monsoon": 189773
  },
  "weather_coverage": 0.03153015795572361,
  "weather_rows": 68280,
  "fire_detections": 1692885
}

QUALITY GATES
  PASS  no duplicate sensor-hours
  PASS  no negative PM2.5
  PASS  all inside India bbox
  FAIL  weather coverage >= 95%
  PASS  has extreme (>=150) rows
  PASS  covers >= 3 seasons
  PASS  includes post_monsoon



## 10. Final outputs

The next cell writes the canonical artifacts consumed by Phase 2 onward:

```text
data/pm25/processed/base/
├── pm25_hourly.parquet        target observations
├── station_manifest.parquet   station selection + exclusion reasons
├── weather_hourly.parquet     canonical meteorology
├── fire_detections.parquet    FIRMS detections
├── base_dataset.parquet       joined, point-in-time-safe dataset
├── data_quality_report.json   gate results
└── dataset_manifest.json      provenance + fingerprint
```

### Stop condition

Do not proceed to Phase 2 until the quality gates pass and the dataset spans
multiple seasons **including `post_monsoon`** — that is the stubble-burning
regime, and a model trained without it cannot predict the episodes AeroPulse
exists to warn about.


In [16]:
# ============================================================================
# FINAL OUTPUTS - canonical dataset + provenance manifest
# ============================================================================
# Everything Phase 2/3/4 needs lands here. dataset_manifest.json records how
# the data was produced, so a model can later be traced back to its inputs.
# ============================================================================
base_path = BASE_DIR / "base_dataset.parquet"
base.to_parquet(base_path, index=False)

dataset_manifest = {
    "schema_version": CONFIG["schema_version"],
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "window_start_utc": str(WINDOW_START),
    "window_end_utc": str(WINDOW_END),
    "requested_window": {"start": CONFIG["start"], "end": CONFIG["end"]},
    "country": CONFIG["country"],
    "sources": {
        "pm25": "OpenAQ v3 /sensors/{id}/hours",
        "weather": "Open-Meteo ERA5 archive",
        "fire": f"NASA FIRMS {FIRMS_SP_SOURCE} + {FIRMS_NRT_SOURCE}",
    },
    "stations_selected": int((~manifest.excluded).sum()),
    "stations_excluded": int(manifest.excluded.sum()),
    "rows": int(len(base)),
    "columns": list(base.columns),
    "fire_radii_km": CONFIG["fire_radii_km"],
    "fire_window_hours": CONFIG["fire_window_hours"],
    "fingerprint": dataset_fingerprint(base[["timestamp_utc", "sensor_id", "pm25"]]),
    "quality_report": report,
    "gates": {k: bool(v) for k, v in GATES.items()},
}
with open(BASE_DIR / "dataset_manifest.json", "w") as f:
    json.dump(dataset_manifest, f, indent=2, default=str)
with open(BASE_DIR / "data_quality_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("PHASE 1 COMPLETE\n")
for p in sorted(BASE_DIR.glob("*")):
    print(f"  {p.name:32} {p.stat().st_size/1e6:8.2f} MB")
print(f"\nBase dataset: {len(base):,} rows x {base.shape[1]} columns")
print(f"Window      : {WINDOW_START:%Y-%m-%d} -> {WINDOW_END:%Y-%m-%d}")
print(f"Stations    : {base.location_id.nunique()}")
print(f"Seasons     : {report['season_coverage']}")
print(f"Fingerprint : {dataset_manifest['fingerprint'][:16]}...")
print("\nPhase 2 reads:", base_path)


PHASE 1 COMPLETE

  base_dataset.parquet                25.65 MB
  data_quality_report.json             0.00 MB
  dataset_manifest.json                0.00 MB
  fire_detections.parquet             30.79 MB
  pm25_hourly.parquet                  5.86 MB
  station_manifest.parquet             0.02 MB
  weather_hourly.parquet               1.06 MB

Base dataset: 1,705,288 rows x 32 columns
Window      : 2025-02-18 -> 2026-09-09
Stations    : 149
Seasons     : {'monsoon': 673073, 'pre_monsoon': 562227, 'winter': 280215, 'post_monsoon': 189773}
Fingerprint : e8d243f781749a0f...

Phase 2 reads: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/base_dataset.parquet


## 11. CPCB backfill adapter (not yet implemented)

OpenAQ cannot supply Indian PM2.5 history before ~Feb 2025. The stub below records the contract for a CPCB CAAQMS backfill without pretending to implement it.


In [ ]:
# ============================================================================
# CPCB BACKFILL ADAPTER (stub) - for the 2018-2025 hole in OpenAQ
# ============================================================================
# WHY THIS EXISTS. OpenAQ's Indian PM2.5 archive has a real gap: major CPCB /
# DPCC stations (R K Puram, Anand Vihar, Punjabi Bagh) hold data for
# 2016-02..2018-02 and then nothing until 2025-02. Verified per-sensor against
# the live API. The median Indian sensor's first observation is 2025-02-18, so
# no OpenAQ-only pipeline can produce a genuine 2-3 year Indian history today.
#
# CPCB CAAQMS is the authoritative source for that period and should be the
# preferred ground truth per the LLD. It has no open REST API - access is via
# the CPCB portal export or a data-sharing agreement - so this is deliberately
# left as a contract, not a fake implementation.
#
# TO IMPLEMENT: emit a frame with exactly the columns normalize_pm25() returns,
# concatenate it with pm25_all before the persist cell, and set
# source="CPCB" so provenance stays traceable per row.
# ============================================================================
CPCB_ENABLED = os.getenv("PM25_ENABLE_CPCB", "0") == "1"


def fetch_cpcb_hourly(station_id, start, end):
    """Return CPCB hourly PM2.5 in AeroPulse's canonical column contract.

    Required columns (match normalize_pm25 exactly):
        timestamp_utc, location_id, sensor_id, station_name, lat, lon,
        pm25, pm25_unit, source, pm25_extreme_flag, pm25_severe_flag
    """
    raise NotImplementedError(
        "CPCB adapter not implemented. Supply a portal export or an agreed API "
        "feed, map it to the canonical columns above, then set PM25_ENABLE_CPCB=1."
    )


if CPCB_ENABLED:
    raise NotImplementedError("PM25_ENABLE_CPCB=1 but fetch_cpcb_hourly() is still a stub.")
print("CPCB backfill adapter: stub registered, disabled "
      f"(PM25_ENABLE_CPCB={os.getenv('PM25_ENABLE_CPCB', '0')})")
print("OpenAQ gap to backfill: ~2018-02 .. 2025-02 for major CPCB/DPCC stations.")
